[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/benjamalegni/ML-luka/blob/main/01-machine-learning/02-clasificacion-palmer-penguins/TP_2a_Preprocesamiento_Datos_EDA.ipynb)


Instalacion dependencias pip

In [ ]:
!pip install ydata-profiling

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 3.2 MB/s eta 0:00:00


In [ ]:
# Obtiene el dataset desde OpenML
from sklearn.datasets import fetch_openml

dataset = fetch_openml(data_id=43347)
dataset.data

/usr/local/lib/python3.12/dist-packages/sklearn/datasets/_openml.py:110: UserWarning: A network error occurred while downloading https://api.openml.org/api/v1/json/data/43347. Retrying...
  warn(


HTTPError: HTTP Error 504: Gateway Time-out

# Analisis exploratorio de datos

Creacion del DataFrame de pandas para generar las estadisticas descriptivas del dataset.
head(10) muestra las ultimas 10 entradas de la tabla

In [ ]:
import pandas as pd

# este codigo comentado solo serviria cargando el dataset de esta forma:
# dataset = openml.datasets.get_dataset(...)
# como use el metodo fetch a openml, cargo el datagram con pd.DataFrame(dataset.data)

#X = dataset.get_data(
#
#    dataset_format='dataframe',
#    target=dataset.default_target_attribute
#)



df = pd.DataFrame(dataset.data)
# df.head(10)
df.describe()

Visualizacion con seaborn
hue="species" para diferenciar datos entre tipos de pinguinos.

Veo que muchos pinguinos con poco peso tiene faltante el valor de sexo. Asi que sera un punto a trabajar en el pre-procesamiento.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

iris_plot = sns.pairplot(df , hue="species", kind="scatter") # kind=scatter kde hist
#iris_plot.map_upper(hide_current_axis)

plt.show()

In [ ]:
from ydata_profiling import ProfileReport

profile = ProfileReport(df)
profile

# Aplicar tareas de preprocesamiento

1. SimpleImputer - tratamiento de valores de "sexo" faltantes.

Los valores de peso a los que les falta asignar sexo corresponderian a los de un pinguino hembra, asi que asigno esos valores de NaN a la constante female.

In [ ]:
import numpy as np
from sklearn.impute import SimpleImputer

# asigno 'female' como la constante para rellenar los valores faltantes.
imp_sex = SimpleImputer(missing_values=np.nan, strategy='constant', fill_value='female')
imp_sex.fit(df[["sex"]])

# .ravel() convierte un array multidimensional en 1-dimensional
df['sex'] = imp_sex.transform(df[["sex"]]).ravel()

df

2. Eliminar aquellas entradas que tienen el atributo body_mass_g = NaN (hay 2 casos)

In [ ]:
df_no_nan = df.dropna(subset=['body_mass_g'], axis=0)
df_no_nan = df_no_nan.reset_index(drop=True)

# count muestra solo entradas no null de la tabla
print(f"filas del dataframe antes de la modificacion:  {len(df)}")
print(f"filas del dataframe despues de la modificacion: {len(df_no_nan)}")

print(f"numero de entradas no null dataframe: \n {df.count()}")
print(f"numero de entradas no null dataframe no nan: \n {df_no_nan.count()}")

In [ ]:
profile_no_nan = ProfileReport(df_no_nan)
profile_no_nan

3. Transformar valores "species","island" y "sex" al primer caracter del atributo.

In [ ]:
# luego intentare aplicar un pipeline
# from sklearn.pipeline import make pipeline
from sklearn.preprocessing import OneHotEncoder

X_species = [
    ["Adelie", "a"],
    ["Chinstrap", "c"],
    ["Gentoo", "g"]
]

X_island = [
    ["Biscoe", "b"],
    ["Dream", "d"],
    ["Torgersen", "t"]
]

X_sex = [
    ["female", "f"],
    ["male", "m"]
]

enc_species = OneHotEncoder(handle_unknown="ignore")
enc_island = OneHotEncoder(handle_unknown="ignore")
enc_sex = OneHotEncoder(handle_unknown="ignore")

df_onehot = df_no_nan.copy()

map_species = dict(X_species)
map_island = dict(X_island)
map_sex = dict(X_sex)

df_onehot["species"] = df_onehot["species"].map(map_species)
df_onehot["island"] = df_onehot["island"].map(map_island)
df_onehot["sex"] = df_onehot["sex"].map(map_sex)

df_onehot

In [ ]:
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Las variables X_species, X_island, X_sex ya están definidas en el entorno global
# Extraemos solo los nombres de las categorías para el OneHotEncoder
species_categories = [item[0] for item in X_species]
island_categories = [item[0] for item in X_island]
sex_categories = [item[0] for item in X_sex]

categorical_features = ['species', 'island', 'sex']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat_ohe', OneHotEncoder(handle_unknown='ignore', categories=[
            species_categories,
            island_categories,
            sex_categories
        ]), categorical_features)
    ],
    remainder='passthrough' # mantiene las columnas no especificadas (numéricas) como están
)

pipe = make_pipeline(preprocessor)
df_pipeline = pipe.fit_transform(df_no_nan)
df_pipeline

notas/preguntas:
- debo crear un nuevo dataframe por cada tecnica de procesamiento que aplico. solo en algunas?